URL index: [Home](https://zzz.bwh.harvard.edu/luna-walkthrough/) | [Data](https://zzz.bwh.harvard.edu/luna-walkthrough/data/) | [S1. File QC](https://zzz.bwh.harvard.edu/luna-walkthrough/p1/) | [S2. Signal QC](https://zzz.bwh.harvard.edu/luna-walkthrough/p2) | [S3. Staging](https://zzz.bwh.harvard.edu/luna-walkthrough/p3) | [S4. Artifacts](https://zzz.bwh.harvard.edu/luna-walkthrough/p4) | [S5. Analysis](https://zzz.bwh.harvard.edu/luna-walkthrough/p5)

Notebook index: [Index](../00_index.ipynb) | [S1. File QC](../p1/00_index.ipynb) | [S2. Signal QC](../p2/00_index.ipynb) | [S3. Staging](../p3/00_index.ipynb) | [S4. Artifacts](../p4/00_index.ipynb) | [S5. Analysis](../p5/00_index.ipynb)

---

# Step 2: signal-level QC

Walkthrough URL = [https://zzz.bwh.harvard.edu/luna-walkthrough/p2/](https://zzz.bwh.harvard.edu/luna-walkthrough/p2/)

In [1]:
import lunapi as lp
proj = lp.proj()

initiated lunapi v1.2.3 <lunapi.lunapi0.luna object at 0x1221e4ab0> 



Having completed Step 1, the `harm1.lst` sample list should now describe a dataset that has consistent channel and annotation labels, consistent annotation formats (standard `.annot`) and nominally consistent units, sample rates and referencing. We can now move on to look at the contents of the EDFs:

 - to describe and handle [recordings with gaps](01_edfd.ipynb)

 - to identify likely [flat or duplicate signals](02_dupes.ipynb)

 - to generate some basic [signal summary statistics](03_stats.ipynb)

 - to spot and fix issues with [EEG polarity](04_pol.ipynb)


We'll first check the channel and annotation labels are as expected:

In [2]:
proj.sample_list( '../harm1.lst' )

read 20 individuals from ../harm1.lst


In [3]:
# luna harm1.lst -o out.db -s ' HEADERS & ANNOTS '
res = proj.proc( 'HEADERS & ANNOTS' )

___________________________________________________________________
Processing: F01 | ../work/harm1/F01.edf
 duration 03.00.30, 10830s | time 22.00.00 - 04.58.00 | date 01.01.85

 signals: 58 (of 58) selected in an EDF+D file
  Fp1 | Fp2 | AF3 | AF4 | F7 | F5 | F3 | F1
  F2 | F4 | F6 | F8 | FT7 | FC5 | FC3 | FC1
  FC2 | FC4 | FC6 | FT8 | T7 | C5 | C3 | C1
  C2 | C4 | C6 | T8 | TP7 | CP5 | CP3 | CP1
  CP2 | CP4 | CP6 | TP8 | P7 | P5 | P3 | P1
  P2 | P4 | P6 | P8 | PO3 | PO4 | O1 | O2
  AFZ | FZ | FCZ | CZ | CPZ | PZ | POz | OZ
  FPZ | EDF Annotations
  extracting 'EDF Annotations' track from EDF+
 ..................................................................
 CMD #1: HEADERS
   options: sig=*
 ..................................................................
 CMD #2: ANNOTS
   options: sig=*
  set epochs to default 30 seconds, 361 epochs
  keeping annotations based on any overlap with an unmasked region
___________________________________________________________________
Processing

We'll extract a table of unique channel/sample-rate/unit combinations present:

In [4]:
# destrat out.db +HEADERS -r CH -v SR PDIM | awk ' NR != 1 { print $2 } ' | sort | uniq -c
proj.table( 'HEADERS' , 'CH' )[[ 'CH' , 'SR', 'PDIM' ]]  

,CH,SR,PDIM
0,AF3,128.0,uV
1,AF3,128.0,uV
2,AF3,128.0,uV
3,AF3,128.0,uV
4,AF3,128.0,uV
...,...,...,...
1134,TP8,128.0,uV
1135,TP8,128.0,uV
1136,TP8,128.0,uV
1137,TP8,128.0,uV


We can use some basic pandas functions to enumerate each type:

In [5]:
tbl = proj.table( 'HEADERS' , 'CH' )[[ 'CH' , 'SR', 'PDIM' ]]  

import pandas as pd
with pd.option_context('display.max_rows', None,):
    display( tbl.groupby(['CH','SR','PDIM']).size().reset_index(name='Count') )

,CH,SR,PDIM,Count
0,AF3,128.0,uV,20
1,AF4,128.0,uV,20
2,AFZ,128.0,uV,20
3,C1,128.0,uV,20
4,C2,128.0,uV,20
5,C3,128.0,uV,20
6,C4,128.0,uV,20
7,C5,128.0,uV,20
8,C6,128.0,uV,20
9,CP1,128.0,uV,20


We see that with one exception (CZ), all channels are seen in all individuals, with the same set of labels applied; CZ appears only in 19 of 20. 

For annotations, we can confirm that all labels have been mapped: 

In [6]:
# destrat out.db +ANNOTS -r ANNOT | awk ' NR != 1 { print $2 } ' | sort | uniq -c
tbl = proj.table( 'ANNOTS' , 'ANNOT' )[[ 'ANNOT' ]]  
tbl.groupby(['ANNOT']).size().reset_index(name='Count')

,ANNOT,Count
0,Arousal,10
1,N1,19
2,N2,20
3,N3,19
4,R,19
5,W,19


That is, we see six distinct annotation labels; only half the sample has `Arousal` annotations marked, but that reflects the original `v2` data, rather than an issue with the steps we've run. 

We'll now move to the [next section](01_edfd.ipynb) that looks at _gapped_ EDF+D files.